**Full environment, dataset, and code setup**

In [ ]:
# =========================================================
# CELL 1 - FULL SETUP COLAB
# =========================================================
# ===== CHANGE DIRECTORY TO /content =====
%cd /content
# ===== CHECK GPU =====
import torch

print("=" * 50)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("=" * 50)

# =========================================================
# CLONE GITHUB
# =========================================================

!git clone https://github.com/ThongLuc2k3/PGA_Unet2D.git

# =========================================================
# DOWNLOAD DATASET ZIP FROM GOOGLE DRIVE
# =========================================================

!gdown --id 1sfMPFQvADmZLCPJC3xPyYDrblnFZ4kQv

# =========================================================
# UNZIP DATASET
# =========================================================

!unzip -q dataset_FracAtlas.zip

# =========================================================
# INSPECT DATASET
# =========================================================

print("\nDATASET STRUCTURE:")
!ls dataset_FracAtlas

# =========================================================
# COPY DATASET INTO THE PROJECT
# =========================================================

!mv dataset_FracAtlas PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/

# =========================================================
# ENTER THE PROJECT DIRECTORY
# =========================================================

%cd PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation

# =========================================================
# INSTALL REQUIREMENTS
# =========================================================

!pip install -q tqdm opencv-python matplotlib scikit-image gdown

print("\nSETUP DONE!")

In [ ]:
%%writefile train_attunet.py
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import logging
import datetime
import numpy as np
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
import random

from models.networks.attention_unet_2D import Attention_UNet_2D

DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE     = 4
EPOCHS         = 100
LR             = 1e-4
WEIGHT_DECAY   = 1e-4
IMG_SIZE       = 512
PATIENCE       = 15
SCHED_PATIENCE = 5

class FracAtlasImageMaskDataset(Dataset):
    """Image-level FracAtlas dataset using the same resize+padding policy as PGA."""
    def __init__(self, image_dir, mask_dir, img_size=512, is_train=True):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.is_train = is_train
        self.images = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    def __len__(self):
        return len(self.images)

    def _resize_and_pad(self, array, interpolation, pad_value=0):
        orig_h, orig_w = array.shape[:2]
        scale = min(self.img_size / orig_w, self.img_size / orig_h)
        new_w = max(1, int(round(orig_w * scale)))
        new_h = max(1, int(round(orig_h * scale)))
        resized = cv2.resize(array, (new_w, new_h), interpolation=interpolation)
        padded = np.full((self.img_size, self.img_size), pad_value, dtype=resized.dtype)
        pad_left = (self.img_size - new_w) // 2
        pad_top = (self.img_size - new_h) // 2
        padded[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized
        return padded

    def __getitem__(self, idx):
        img_name = self.images[idx]
        image = cv2.imread(os.path.join(self.image_dir, img_name), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(os.path.join(self.mask_dir, img_name), cv2.IMREAD_GRAYSCALE)
        image = self._resize_and_pad(image, cv2.INTER_LINEAR, pad_value=0)
        mask = self._resize_and_pad(mask, cv2.INTER_NEAREST, pad_value=0)
        image = (image.astype(np.float32) / 255.0 - 0.5) / 0.5
        mask = (mask > 127).astype(np.float32)
        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)
        if self.is_train:
            if random.random() >= 0.5:
                image, mask = TF.hflip(image), TF.hflip(mask)
            if random.random() >= 0.5:
                angle = random.uniform(-15, 15)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
        mask = (mask > 0.5).float()
        return image, mask


def dice_loss(pred, target, smooth=1e-5):
    pred_soft = torch.sigmoid(pred)
    intersection = (pred_soft * target).sum(dim=(1, 2, 3))
    union = pred_soft.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return (1 - (2. * intersection + smooth) / (union + smooth)).mean()


def batch_metrics_sum(pred, target, smooth=1e-5):
    pred_bin = (torch.sigmoid(pred) > 0.5).float()
    tp = (pred_bin * target).sum(dim=(1, 2, 3))
    fp = (pred_bin * (1 - target)).sum(dim=(1, 2, 3))
    fn = ((1 - pred_bin) * target).sum(dim=(1, 2, 3))
    dice = (2. * tp + smooth) / (2. * tp + fp + fn + smooth)
    iou = (tp + smooth) / (tp + fp + fn + smooth)
    precision = tp / (tp + fp + smooth)
    recall = tp / (tp + fn + smooth)
    return dice.sum().item(), iou.sum().item(), precision.sum().item(), recall.sum().item()


def setup_logger():
    os.makedirs("logs", exist_ok=True)
    t = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    logging.basicConfig(
        level=logging.INFO, format='%(message)s',
        handlers=[logging.FileHandler(f"logs/train_attunet_{t}.log", encoding='utf-8'), logging.StreamHandler()]
    )
    return logging.getLogger()


def main():
    logger = setup_logger()
    logger.info("=" * 90)
    logger.info(f"TRAIN ATTENTION U-NET 2D | Device: {DEVICE}")
    logger.info(f"Batch: {BATCH_SIZE} | MaxEpochs: {EPOCHS} | LR: {LR} | ImgSize: {IMG_SIZE}")
    logger.info(f"WeightDecay: {WEIGHT_DECAY} | EarlyStop patience: {PATIENCE}")
    logger.info("=" * 90)

    train_ds = FracAtlasImageMaskDataset("dataset_FracAtlas/train/images", "dataset_FracAtlas/train/masks", img_size=IMG_SIZE, is_train=True)
    val_ds = FracAtlasImageMaskDataset("dataset_FracAtlas/val/images", "dataset_FracAtlas/val/masks", img_size=IMG_SIZE, is_train=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = Attention_UNet_2D(in_channels=1, n_classes=1).to(DEVICE)
    criterion_bce = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=SCHED_PATIENCE, min_lr=1e-7)

    os.makedirs("checkpoints", exist_ok=True)
    best_val_dice = 0.0
    no_improve = 0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
        for images, masks in loop:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            preds = model(images)
            loss = criterion_bce(preds, masks) + dice_loss(preds, masks)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            loop.set_postfix(loss=f"{loss.item():.4f}")

        model.eval()
        sum_dice = sum_iou = sum_pre = sum_rec = 0.0
        total = 0
        with torch.no_grad():
            for vi, vm in val_loader:
                vi, vm = vi.to(DEVICE), vm.to(DEVICE)
                vout = model(vi)
                d, i, p, r = batch_metrics_sum(vout, vm)
                sum_dice += d; sum_iou += i; sum_pre += p; sum_rec += r
                total += vi.size(0)

        val_dice = sum_dice / total
        scheduler.step(val_dice)
        log_str = (f"Epoch {epoch+1:3d} | Loss: {train_loss/len(train_loader):.4f} | "
                   f"Dice: {val_dice:.4f} | IoU: {sum_iou/total:.4f} | "
                   f"Pre: {sum_pre/total:.4f} | Rec: {sum_rec/total:.4f} | "
                   f"LR: {optimizer.param_groups[0]['lr']:.2e}")
        torch.save(model.state_dict(), "checkpoints/attunet_last.pth")
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            no_improve = 0
            torch.save(model.state_dict(), "checkpoints/att_unet_best.pth")
            log_str = "[BEST] " + log_str
        else:
            no_improve += 1
        logger.info(log_str)
        if no_improve >= PATIENCE:
            logger.info(f"Early stopping at epoch {epoch+1}.")
            break

    logger.info(f"
Best Dice: {best_val_dice:.4f}")
    logger.info("Checkpoint: checkpoints/att_unet_best.pth")

if __name__ == "__main__":
    main()


**Train**

In [ ]:
# =========================================================
# TRAIN
# =========================================================

!python train_attunet.py


**Test + Dice + IoU + BCE**

ATT_Unet2D

In [ ]:
# @title
# =========================================================
# TEST PHASE (ATTENTION U-NET 2D - METRICS SYNCHRONIZED WITH PROMPT-UNET)
# =========================================================

import os
import cv2
import torch
import random
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from scipy.ndimage import binary_erosion, distance_transform_edt
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

from models.networks.attention_unet_2D import Attention_UNet_2D

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "checkpoints/att_unet_best.pth"
IMG_SIZE = 512

class FracAtlasImageMaskDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=512, is_train=False):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.is_train = is_train
        self.images = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    def __len__(self):
        return len(self.images)

    def _resize_and_pad(self, array, interpolation, pad_value=0):
        orig_h, orig_w = array.shape[:2]
        scale = min(self.img_size / orig_w, self.img_size / orig_h)
        new_w = max(1, int(round(orig_w * scale)))
        new_h = max(1, int(round(orig_h * scale)))
        resized = cv2.resize(array, (new_w, new_h), interpolation=interpolation)
        padded = np.full((self.img_size, self.img_size), pad_value, dtype=resized.dtype)
        pad_left = (self.img_size - new_w) // 2
        pad_top = (self.img_size - new_h) // 2
        padded[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized
        return padded

    def __getitem__(self, idx):
        img_name = self.images[idx]
        image = cv2.imread(os.path.join(self.image_dir, img_name), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(os.path.join(self.mask_dir, img_name), cv2.IMREAD_GRAYSCALE)
        image = self._resize_and_pad(image, cv2.INTER_LINEAR, pad_value=0)
        mask = self._resize_and_pad(mask, cv2.INTER_NEAREST, pad_value=0)
        image = (image.astype(np.float32) / 255.0 - 0.5) / 0.5
        mask = (mask > 127).astype(np.float32)
        return torch.from_numpy(image).unsqueeze(0), torch.from_numpy(mask).unsqueeze(0)


def extract_lcc(binary_map: np.ndarray) -> np.ndarray:
    if binary_map.sum() == 0:
        return binary_map
    mask_uint8 = binary_map.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_uint8, connectivity=8)
    if num_labels <= 1:
        return binary_map
    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (labels == largest_label).astype(np.float32)


def calc_hd95(pred: np.ndarray, gt: np.ndarray) -> float:
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or not gt.any(): return float(IMG_SIZE)
    pe = pred ^ binary_erosion(pred)
    ge = gt ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]
    d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))


def calc_cbl(pred_bin: np.ndarray, gt_bin: np.ndarray):
    if gt_bin.sum() == 0: return None
    ys, xs = np.where(gt_bin)
    gt_diag = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + 1e-6
    if pred_bin.sum() == 0: return 0.0
    yp, xp = np.where(pred_bin)
    d = np.sqrt((xp.mean()-xs.mean())**2 + (yp.mean()-ys.mean())**2)
    return float(np.clip(1.0 - d/gt_diag, 0.0, 1.0))


def get_centroid(binary_map: np.ndarray):
    if binary_map.sum() == 0: return None, None
    ys, xs = np.where(binary_map)
    return float(xs.mean()), float(ys.mean())

model = Attention_UNet_2D(in_channels=1, n_classes=1).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()

test_dataset = FracAtlasImageMaskDataset("dataset_FracAtlas/test/images", "dataset_FracAtlas/test/masks", img_size=IMG_SIZE, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

SHOW_INDEX = list(range(10))
all_dice, all_iou, all_pre, all_rec, all_hd95, all_cbl = [], [], [], [], [], []
smooth = 1e-5

with torch.no_grad():
    for idx, (images, masks) in enumerate(test_loader):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        outputs = model(images)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        img_np = (images[0, 0].cpu().numpy() + 1) / 2.0
        gm = masks[0, 0].cpu().numpy()
        pm = preds[0, 0].cpu().numpy()
        pm = extract_lcc(pm)
        tp = (pm * gm).sum()
        fp = (pm * (1 - gm)).sum()
        fn = ((1 - pm) * gm).sum()
        dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
        iou = (tp + smooth) / (tp + fp + fn + smooth)
        pre = (tp + smooth) / (tp + fp + smooth)
        rec = (tp + smooth) / (tp + fn + smooth)
        hd = calc_hd95(pm.astype(bool), gm.astype(bool))
        cbl = calc_cbl(pm.astype(bool), gm.astype(bool))
        all_dice.append(dice); all_iou.append(iou); all_pre.append(pre); all_rec.append(rec); all_hd95.append(hd)
        if cbl is not None: all_cbl.append(cbl)

        if idx in SHOW_INDEX:
            cx_gt, cy_gt = get_centroid(gm)
            cx_p, cy_p = get_centroid(pm)
            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            fig.suptitle(f"Attention U-Net test | Sample ID: {idx}", fontsize=14, fontweight='bold')
            axes[0].imshow(img_np, cmap='gray'); axes[0].set_title("Input image", fontsize=11, fontweight='bold')
            axes[1].imshow(img_np, cmap='gray')
            green = np.zeros((*gm.shape, 4), dtype=np.float32); green[gm == 1] = [0, 1, 0, 0.3]
            axes[1].imshow(green)
            if gm.max() > 0: axes[1].contour(gm, [0.5], colors='lime', linewidths=1.5)
            if cx_gt is not None:
                axes[1].plot(cx_gt, cy_gt, 'o', color='lime', ms=8, alpha=1.0, markeredgecolor='black', label='GT centroid')
                axes[1].legend(loc='lower right', fontsize=8)
            axes[1].set_title("Ground truth", fontsize=11, fontweight='bold')
            axes[2].imshow(img_np, cmap='gray')
            red = np.zeros((*pm.shape, 4), dtype=np.float32); red[pm == 1] = [1, 0, 0, 0.3]
            axes[2].imshow(red)
            if pm.max() > 0: axes[2].contour(pm, [0.5], colors='red', linewidths=1.5)
            if cx_p is not None:
                axes[2].plot(cx_p, cy_p, 'o', color='red', ms=8, alpha=1.0, markeredgecolor='white', label='Predicted centroid')
                axes[2].legend(loc='lower right', fontsize=8)
            axes[2].set_title("Prediction (Att-UNet)", fontsize=11, fontweight='bold')
            axes[3].imshow(img_np, cmap='gray')
            if gm.max() > 0: axes[3].contour(gm, [0.5], colors='lime', linewidths=2)
            if pm.max() > 0: axes[3].contour(pm, [0.5], colors='red', linewidths=2, linestyles='--')
            if cx_gt is not None: axes[3].plot(cx_gt, cy_gt, 'o', color='lime', ms=8, alpha=1.0, markeredgecolor='black')
            if cx_p is not None: axes[3].plot(cx_p, cy_p, 'o', color='red', ms=8, alpha=1.0, markeredgecolor='white')
            if cx_gt is not None and cx_p is not None: axes[3].plot([cx_gt, cx_p], [cy_gt, cy_p], '--', color='yellow', lw=1.5, alpha=1.0)
            axes[3].set_title(f"Dice: {dice:.3f} | IoU: {iou:.3f} | HD95: {hd:.1f}px
CBL: {cbl:.3f} | Pre: {pre:.3f} | Rec: {rec:.3f}", fontsize=10, fontweight='bold')
            for ax in axes: ax.axis('off')
            plt.tight_layout(); plt.show()

print("
" + "=" * 60)
print("📊 FINAL TEST RESULTS - ATTENTION U-NET")
print("=" * 60)
print(f"Mean Dice ↑      : {np.mean(all_dice):.4f}")
print(f"Mean IoU ↑       : {np.mean(all_iou):.4f}")
print(f"Mean Precision ↑ : {np.mean(all_pre):.4f}")
print(f"Mean Recall ↑    : {np.mean(all_rec):.4f}")
print(f"Mean HD95 ↓ (px) : {np.mean(all_hd95):.2f}")
print(f"Mean CBL ↑       : {np.mean(all_cbl):.4f}")
print(f"Total Samples    : {len(all_dice)}")
print("=" * 60)
import csv
os.makedirs("results", exist_ok=True)
csv_path = "results/attunet2d_results.csv"
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["model", "dice", "iou", "precision", "recall", "hd95", "cbl", "n_samples"])
    writer.writerow(["AttUNet2D", f"{np.mean(all_dice):.4f}", f"{np.mean(all_iou):.4f}", f"{np.mean(all_pre):.4f}", f"{np.mean(all_rec):.4f}", f"{np.mean(all_hd95):.4f}", f"{np.mean(all_cbl):.4f}", len(all_dice)])
print(f"
Results saved: {csv_path}")
